# Kalman Filters: Dynamic Hedge Ratios

**pyportfolios.com tutorial T14** · EWA / EWC, Jan 2010 – Dec 2024 · NumPy · statsmodels · Pandas

A pairs trade is only as good as its hedge ratio — and hedge ratios drift. In this
notebook we

1. estimate the EWC~EWA hedge ratio with full-sample OLS and rolling 252-day OLS,
2. build a Kalman filter **from scratch in NumPy** that treats beta and alpha as
   random-walk states,
3. compare the three beta paths, and
4. trade the identical z-score rules on the Kalman spread vs the static spread,
   with 10 bp costs on both legs.

Everything is deterministic — no random numbers anywhere.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import yfinance as yf
from statsmodels.regression.rolling import RollingOLS
from statsmodels.tsa.stattools import adfuller

plt.rcParams["figure.figsize"] = (10, 5)

## 1 · Data: the classic Chan pair

EWA (iShares MSCI Australia) and EWC (iShares MSCI Canada) — two commodity-heavy,
developed-market ETFs made famous as a cointegration example by Ernest Chan.
Fifteen years of adjusted closes, 2010–2024.

In [ ]:
px = yf.download(["EWA", "EWC"], start="2010-01-01", end="2025-01-01",
                 auto_adjust=True, progress=False)["Close"].dropna()
ewa, ewc = px["EWA"].to_numpy(), px["EWC"].to_numpy()

(px / px.iloc[0]).plot(title="EWA & EWC, normalized");

## 2 · Two OLS baselines

The full-sample regression `EWC ~ EWA` gives one number for 15 years — and it is
computed with data you would not have had for most of the sample. The rolling
252-day OLS is honest but laggy: every observation inside the window has
equal weight, so a year-old data point moves today's beta exactly as much as
yesterday's, and points falling *out* of the window jerk the estimate (the
"window cliff").

In [ ]:
x_const = sm.add_constant(px["EWA"])

ols = sm.OLS(px["EWC"], x_const).fit()
beta_static, alpha_static = float(ols.params["EWA"]), float(ols.params["const"])

spread_static = px["EWC"] - beta_static * px["EWA"] - alpha_static
print(f"static beta  = {beta_static:.4f}   alpha = {alpha_static:.4f}")
print(f"ADF p-value on the static spread = {adfuller(spread_static.to_numpy())[1]:.4f}")

roll = RollingOLS(px["EWC"], x_const, window=252).fit(params_only=True)
beta_roll = roll.params["EWA"].to_numpy()
alpha_roll = roll.params["const"].to_numpy()

## 3 · The Kalman filter, from scratch

State-space model — the state is the hedge relationship itself:

$$\begin{aligned}
\text{state:} \quad & \begin{bmatrix}\beta_t \\ \alpha_t\end{bmatrix}
 = \begin{bmatrix}\beta_{t-1} \\ \alpha_{t-1}\end{bmatrix} + \omega_t,
 \qquad \omega_t \sim \mathcal N(0, Q) \\
\text{observation:} \quad & EWC_t = \beta_t \, EWA_t + \alpha_t + \varepsilon_t,
 \qquad \varepsilon_t \sim \mathcal N(0, R)
\end{aligned}$$

A random walk for the states (transition matrix $F = I$) says: *the hedge ratio
tomorrow is the hedge ratio today, plus noise.* The standard parameterization
(Chan 2013) sets $Q = \frac{\delta}{1-\delta} I$ with $\delta = 1e-05$
and $R = 0.001$. Delta is the knob: larger → beta adapts faster but is noisier;
smaller → smoother but laggier. $\delta = 0$ recovers recursive least squares
(a static beta refined forever).

In [ ]:
def kalman_hedge(x, y, delta=1e-05, r_obs=0.001):
    n = len(x)
    q = (delta / (1.0 - delta)) * np.eye(2)   # trans_cov
    state = np.zeros(2)                       # [beta, alpha], diffuse start
    p_cov = np.eye(2)
    betas, alphas = np.zeros(n), np.zeros(n)
    for t in range(n):
        h = np.array([x[t], 1.0])             # observation map
        p_cov = p_cov + q                     # predict (F = I)
        e = y[t] - h @ state                  # innovation
        s = h @ p_cov @ h + r_obs             # innovation variance
        k = p_cov @ h / s                     # Kalman gain
        state = state + k * e                 # update
        p_cov = p_cov - np.outer(k, h @ p_cov)
        betas[t], alphas[t] = state
    return betas, alphas

beta_kf, alpha_kf = kalman_hedge(ewa, ewc)
print(f"Kalman beta: first tradeable {beta_kf[252]:.3f} ... last {beta_kf[-1]:.3f}")

## 4 · Three betas, one pair

Slice off the first year (the rolling OLS warm-up / Kalman burn-in) and compare.
The Kalman beta moves *with* the relationship — exponentially down-weighting old
data — while the rolling OLS drags a 252-day anchor and the static line is a
15-year average of regimes that no longer exist.

In [ ]:
t0 = 252
dates = px.index[t0:]
plt.plot(dates, beta_kf[t0:], label="Kalman", lw=1.6)
plt.plot(dates, beta_roll[t0:], label="rolling OLS (252d)", lw=1.2)
plt.axhline(beta_static, color="k", ls="--", lw=1, label="static OLS")
plt.legend(); plt.title("EWC~EWA hedge ratio, three estimators");
plt.show()

## 5 · Trading the spread

Identical rules on both spreads: z-score the spread on a trailing 60-day
window, enter long (long EWC / short beta·EWA) at z < −2, short at
z > +2, exit when z crosses 0. Positions are sized to \$1 gross
notional at entry; the Kalman variant re-hedges the EWA leg to the current beta
each day. 10 bp cost on every unit of traded notional. Signals use the close,
P&L accrues from the next day — no lookahead in the rule (the static beta itself,
of course, is one giant lookahead — that is the point of the comparison).

In [ ]:
def backtest(beta, alpha, dynamic, z_win=60, entry=2,
             cost=0.001, start=252):
    n = len(ewa)
    spread = pd.Series(ewc - beta * ewa - alpha)
    z = ((spread - spread.rolling(z_win).mean())
         / spread.rolling(z_win).std(ddof=1)).to_numpy()

    pos, p = np.zeros(n), 0.0
    for t in range(start, n):
        zt = z[t]
        if np.isnan(zt):
            pos[t] = p; continue
        if p == 0.0:
            if zt < -entry: p = 1.0
            elif zt > entry: p = -1.0
        elif p == 1.0 and zt >= 0.0: p = 0.0
        elif p == -1.0 and zt <= 0.0: p = 0.0
        pos[t] = p

    ret = np.zeros(n)
    n_ewc = n_ewa = g0 = 0.0
    trades = 0
    for t in range(start, n):
        pnl = n_ewc * (ewc[t] - ewc[t-1]) + n_ewa * (ewa[t] - ewa[t-1])
        c = 0.0
        if pos[t] != pos[t-1]:
            if pos[t-1] == 0.0:                     # entry
                g0 = ewc[t] + abs(beta[t]) * ewa[t]
                tc, ta = pos[t] / g0, -pos[t] * beta[t] / g0
                trades += 1
            else:                                   # exit
                tc = ta = 0.0
            c = cost * (abs(tc - n_ewc) * ewc[t] + abs(ta - n_ewa) * ewa[t])
            n_ewc, n_ewa = tc, ta
        elif dynamic and pos[t] != 0.0:             # re-hedge to beta_t
            ta = -pos[t] * beta[t] / g0
            c = cost * abs(ta - n_ewa) * ewa[t]
            n_ewa = ta
        ret[t] = pnl - c

    rr = ret[start:]
    eq = np.cumprod(1.0 + rr)
    ann, vol = rr.mean() * 252, rr.std(ddof=1) * np.sqrt(252)
    mdd = (eq / np.maximum.accumulate(eq) - 1.0).min()
    return eq, dict(ann_ret=ann, ann_vol=vol, sharpe=ann / vol,
                    max_dd=mdd, trades=trades)

eq_kf, st_kf = backtest(beta_kf, alpha_kf, dynamic=True)
eq_st, st_st = backtest(np.full(len(ewa), beta_static),
                        np.full(len(ewa), alpha_static), dynamic=False)

plt.plot(px.index[252:], eq_kf, label="Kalman beta")
plt.plot(px.index[252:], eq_st, label="static beta")
plt.legend(); plt.title("Spread mean reversion, net of 10 bp costs");
plt.show()

## 6 · What the numbers actually say

Run the comparison gross (cost = 0) as well as net, because the two answers
differ — and the difference is the real lesson. The Kalman spread is the better
*signal*: higher gross Sharpe, half the volatility, half the max drawdown. But
it mean-reverts faster, so it trades ~2.7× as often, and at 10 bp per unit of
traded notional the extra turnover eats the entire edge. The static-beta
variant survives 10 bp — while quietly enjoying a 15-year lookahead, since its
beta was fit on the full sample.

In [ ]:
eq_kf_g, st_kf_g = backtest(beta_kf, alpha_kf, dynamic=True, cost=0.0)
eq_st_g, st_st_g = backtest(np.full(len(ewa), beta_static),
                            np.full(len(ewa), alpha_static),
                            dynamic=False, cost=0.0)

rows = pd.DataFrame([st_kf, st_st, st_kf_g, st_st_g],
                    index=["Kalman (10bp)", "static (10bp)",
                           "Kalman (gross)", "static (gross)"])
rows.style.format({"ann_ret": "{:.2%}", "ann_vol": "{:.2%}", "sharpe": "{:.2f}",
                   "max_dd": "{:.2%}"})

## Takeaways

- A hedge ratio is an estimate of a *relationship*, and relationships drift —
  the Kalman filter models the drift instead of averaging over it.
- Five lines of linear algebra (predict, innovate, gain, update, covariance)
  replace a window size with a forgetting rate; delta is the only real knob.
- The rolling OLS lags by construction: equal weights inside the window, a
  cliff at its edge. The filter's exponential weighting has neither problem.
- On EWA/EWC the Kalman spread is the better signal (gross Sharpe 0.55 vs 0.47,
  half the drawdown) but the costlier one to trade: at 10 bp its ~2.7× turnover
  flips the net ranking. Estimation quality and implementability are different
  axes — a backtest that only reports one of them is hiding the other.
- Delta and R were set to textbook values, not tuned on this sample. Tune them
  and you are back in backtest-overfitting territory (see the pairs-trading
  tutorial's caveats).

*© pyportfolios.com — runnable companion to the article. Data: Yahoo Finance via yfinance.*